# Steerling-8B — Concept Inspection & Steering

Make sure you're using the **Steerling (Python 3.13 ARM)** kernel.

In [1]:
import torch
from steerling import SteerlingGenerator, GenerationConfig

DEVICE = "mps"
MODEL_PATH = "/Users/vb/.cache/huggingface/hub/models--guidelabs--steerling-8b/snapshots/337e00164c67b3e458de12430246bd9e633568f7"

print("Loading model from local cache...")
gen = SteerlingGenerator.from_pretrained(MODEL_PATH, device=DEVICE)
print(f"Ready. Interpretable: {gen.is_interpretable}")

Loading model from local cache...
Ready. Interpretable: True


## 1. Basic generation

In [2]:
prompt = "The key to understanding neural networks is"

text = gen.generate(prompt, GenerationConfig(max_new_tokens=100, seed=42))
print(text)

 that they are made up of layers of interconnected nodes. Each node represents a neuron, which can process inputs and generate outputs based on the information given to it.

When it comes to deep learning, the algorithms used in this network have multiple layers of neurons. These hidden layers are responsible for extracting meaningful features from the input data. The final layer of the network is the output layer, where predictions or classifications are generated.

Deep learning models are trained using large amounts of labeled data. This means that the


## 2. Inspect active concepts

The model decomposes its hidden state into 33,732 known concept dimensions.
At each token position the top-16 concept IDs are active. Here we aggregate across
positions (mean activation) to see which concepts dominate for a given input.

In [8]:
def get_concepts(text: str, topk: int = 20) -> list[tuple[int, float]]:
    """Return top-k known concept IDs averaged across token positions."""
    token_ids = gen.tokenizer.encode(text, add_special_tokens=False)
    x = torch.tensor([token_ids], dtype=torch.long, device=gen.device)

    with torch.inference_mode():
        # minimal_output=True forces the streaming top-k path, which populates
        # known_topk_indices/logits. minimal_output=False triggers the dense logit
        # path (return_logits=True) which skips top-k and leaves those fields None.
        _, outputs = gen.model(x, use_teacher_forcing=False, minimal_output=True)

    # shape: (1, T, K) -> (T, K)
    indices = outputs.known_topk_indices[0]
    logits  = outputs.known_topk_logits[0]

    scores: dict[int, list[float]] = {}
    for t in range(indices.shape[0]):
        for k in range(indices.shape[1]):
            cid   = int(indices[t, k].item())
            score = float(logits[t, k].item())
            scores.setdefault(cid, []).append(score)

    averaged = {cid: sum(v) / len(v) for cid, v in scores.items()}
    return sorted(averaged.items(), key=lambda x: x[1], reverse=True)[:topk]

In [9]:
text = "The economy is growing rapidly"
concepts = get_concepts(text, topk=20)

print(f"Top concepts for: '{text}'\n")
for rank, (cid, score) in enumerate(concepts, 1):
    print(f"  #{rank:2d}  [{cid:6d}]  {score:.4f}")

Top concepts for: 'The economy is growing rapidly'

  # 1  [ 12707]  3.0755
  # 2  [ 33644]  2.6571
  # 3  [ 33612]  1.6828
  # 4  [ 19647]  1.4281
  # 5  [ 31191]  0.9196
  # 6  [ 33628]  0.7570
  # 7  [ 28490]  0.4567
  # 8  [  3858]  -0.0437
  # 9  [  9256]  -0.1179
  #10  [ 33622]  -0.1789
  #11  [ 33606]  -0.2399
  #12  [  5641]  -0.3179
  #13  [ 13938]  -0.3485
  #14  [ 26273]  -0.3838
  #15  [ 33728]  -0.4419
  #16  [ 31108]  -0.6550
  #17  [ 33671]  -0.6734
  #18  [ 28312]  -0.8153
  #19  [ 33618]  -0.8640
  #20  [ 31978]  -0.9269


## 3. Compare concepts across two prompts

Which concepts are shared vs. unique between two different texts?

In [10]:
def compare_concepts(text_a: str, text_b: str, topk: int = 15):
    ca = dict(get_concepts(text_a, topk * 2))
    cb = dict(get_concepts(text_b, topk * 2))

    shared = sorted(set(ca) & set(cb), key=lambda c: ca[c] + cb[c], reverse=True)[:topk]
    only_a = sorted(set(ca) - set(cb), key=lambda c: ca[c], reverse=True)[:topk]
    only_b = sorted(set(cb) - set(ca), key=lambda c: cb[c], reverse=True)[:topk]

    print(f"A: '{text_a}'")
    print(f"B: '{text_b}'\n")

    print("── Shared ──────────────────────")
    for c in shared:
        print(f"  [{c:6d}]  A={ca[c]:.3f}  B={cb[c]:.3f}")

    print("\n── Only in A ────────────────────")
    for c in only_a:
        print(f"  [{c:6d}]  {ca[c]:.3f}")

    print("\n── Only in B ────────────────────")
    for c in only_b:
        print(f"  [{c:6d}]  {cb[c]:.3f}")


compare_concepts("The economy is booming", "The economy is collapsing")

A: 'The economy is booming'
B: 'The economy is collapsing'

── Shared ──────────────────────
  [ 33644]  A=1.328  B=2.575
  [ 33612]  A=1.270  B=2.085
  [ 13938]  A=1.654  B=1.108
  [ 18306]  A=0.474  B=0.967
  [  5641]  A=-0.781  B=1.555
  [  3858]  A=-0.278  B=-0.374
  [ 33618]  A=-1.244  B=0.588
  [ 33622]  A=-0.108  B=-1.670
  [ 33671]  A=-1.303  B=-0.697
  [ 33606]  A=-0.820  B=-1.469
  [  8142]  A=-1.204  B=-1.397
  [  7195]  A=-0.811  B=-1.800
  [ 25411]  A=-1.380  B=-1.491
  [ 26527]  A=-1.206  B=-1.681

── Only in A ────────────────────
  [  9862]  0.257
  [  2881]  -0.058
  [  3072]  -0.655
  [ 26970]  -0.664
  [ 24463]  -0.721
  [ 33183]  -0.877
  [  4211]  -1.172
  [  2635]  -1.257
  [  6455]  -1.267
  [ 12707]  -1.460
  [   347]  -1.558
  [ 28551]  -1.582
  [ 32955]  -1.584
  [  8453]  -1.639
  [ 17481]  -1.692

── Only in B ────────────────────
  [ 22176]  4.346
  [ 31191]  -0.361
  [ 27044]  -0.859
  [ 28476]  -0.910
  [ 31108]  -1.017
  [ 31879]  -1.285
  [ 30134]  -1.3

## 4. Concept steering

`steer_known` maps `{concept_id: strength}`. Positive values amplify, negative suppress.
Compare baseline vs. steered generation on the same prompt + seed.

In [11]:
def steer(prompt, steer_known=None, steer_unknown=None, max_new_tokens=80, seed=42):
    cfg = GenerationConfig(
        max_new_tokens=max_new_tokens,
        seed=seed,
        steer_known=steer_known,
        steer_unknown=steer_unknown,
    )
    return gen.generate(prompt, cfg)


prompt = "The economy is"
concepts = get_concepts(prompt, topk=5)
top_id    = concepts[0][0]
second_id = concepts[1][0]

print(f"Top concepts: {concepts[:5]}\n")
print(f"── Baseline ──────────────────────────────")
print(steer(prompt))

print(f"\n── Amplify [{top_id}] x5 ─────────────────")
print(steer(prompt, steer_known={top_id: 5.0}))

print(f"\n── Suppress [{second_id}] x-5 ────────────")
print(steer(prompt, steer_known={second_id: -5.0}))

Top concepts: [(33612, 2.354645848274231), (33644, 1.5699350833892822), (31191, 1.2647826671600342), (33622, 0.7760270833969116), (5641, 0.7729023694992065)]

── Baseline ──────────────────────────────
 based on agriculture and fishing. The island has a small number of inhabitants, most of them living in the capital city.

Geography

Corsica covers an area of ​​about 750 square kilometers and lies about 50 km away from the west coast of France. Corsica consists of two parts: the northern part is called Corsica and the southern part Sardinia. The parts are separated

── Amplify [33612] x5 ─────────────────
       	                  		                                                                                    	                                                                                                                                                                                                                                       	                                          

## 5. Concept activation heatmap (per token)

Show which concepts fire at each token position.

In [12]:
def concept_heatmap(text: str):
    token_ids = gen.tokenizer.encode(text, add_special_tokens=False)
    tokens = [gen.tokenizer.decode([tid]) for tid in token_ids]
    x = torch.tensor([token_ids], dtype=torch.long, device=gen.device)

    with torch.inference_mode():
        _, outputs = gen.model(x, use_teacher_forcing=False, minimal_output=True)

    indices = outputs.known_topk_indices[0].cpu()  # (T, K)
    logits  = outputs.known_topk_logits[0].cpu()   # (T, K)

    print(f"Token-level top-3 concept activations for: '{text}'\n")
    for t, tok in enumerate(tokens):
        top3 = [(int(indices[t, k]), float(logits[t, k])) for k in range(3)]
        concept_str = "  ".join(f"[{cid}]={s:.2f}" for cid, s in top3)
        print(f"  {tok!r:20s}  {concept_str}")


concept_heatmap("Artificial intelligence is transforming medicine")

Token-level top-3 concept activations for: 'Artificial intelligence is transforming medicine'

  'Art'                 [33622]=3.24  [33644]=3.00  [33612]=1.57
  'ificial'             [33622]=1.08  [28551]=0.33  [15697]=-0.18
  ' intelligence'       [33618]=1.36  [12020]=1.27  [33612]=-0.28
  ' is'                 [33618]=0.84  [25411]=-1.19  [533]=-2.67
  ' transforming'       [33612]=5.50  [33618]=2.16  [33644]=1.48
  ' medicine'           [25047]=3.20  [14152]=0.50  [4211]=-0.16


## 6. Sandbox — try your own steering

Pick any concept IDs from the inspection cells above and steer at different strengths.

In [13]:
MY_PROMPT = "Climate change is"

# Step 1: see what concepts are active
for cid, score in get_concepts(MY_PROMPT, topk=10):
    print(f"  [{cid:6d}]  {score:.4f}")

  [ 33612]  5.9303
  [ 33644]  5.1746
  [ 33671]  1.1744
  [ 31191]  0.3538
  [  5641]  -0.0194
  [ 19244]  -0.2738
  [ 33606]  -0.3664
  [  2842]  -0.6634
  [ 33618]  -0.7194
  [ 33622]  -0.7740


In [14]:
# Step 2: steer — edit concept_id and strength
concept_id = 0      # replace with an ID from above
strength   = 5.0    # positive = amplify, negative = suppress

print("Baseline:")
print(steer(MY_PROMPT))

print(f"\nSteered [{concept_id}] x{strength}:")
print(steer(MY_PROMPT, steer_known={concept_id: strength}))

Baseline:
 also likely to increase the incidence of infectious diseases, such as dengue fever and malaria. This is because warmer temperatures will allow mosquitoes to thrive in areas where they were previously unable to survive.

Overall, climate change is expected to have a significant impact on human health. It is important that we take action now to reduce our greenhouse gas emissions and adapt to the changing climate so that we can minimize

Steered [0] x5.0:
 block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block blocks block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block block
